# EA3 — Procesamiento distribuido de datos con Apache Spark

**Big Data (ISD-25)** · Ingeniería de Software y Datos · IU Digital de Antioquia

| | |
|---|---|
| **Grupo** | 63 |
| **Integrantes** | Jorge Andrés Ocampo Suárez |
| **Caso de estudio** | Wanderbricks |
| **Fecha de entrega** | domingo 20 de septiembre |
| **🎥 Enlace al video** | *(pegar aquí — 9 a 12 minutos, mínimo 4 minutos por integrante)* |

> ⚠️ **Antes de entregar:** verificar que el enlace del video abra desde una cuenta distinta a la propia.
> Un enlace inaccesible se califica como no entregado.

---
## 1. Contexto y problema

*Qué preguntas debe responder este pipeline y por qué requiere procesamiento distribuido.*

---
## 2. Descripción de los datos

**Requisito de volumen:** mínimo 20 millones de registros en la tabla principal,
y al menos dos tablas de más de un millón de filas cada una que se crucen entre sí.

*Justificar el camino elegido: `samples.tpch` (lineitem tiene 29.999.795 filas),
`samples.tpcds_sf1`, o amplificación sintética del clickstream conservando su esquema real.
`samples.nyctaxi` no sirve: solo tiene 21.932 filas.*

In [0]:
# Verificación del volumen con el que van a trabajar
TABLA_PRINCIPAL = "samples.tpch.lineitem" 
TABLA_SECUNDARIA = "samples.tpch.orders"

for t in [TABLA_PRINCIPAL, TABLA_SECUNDARIA]:
    print(f"{t:35s} {spark.table(t).count():>12,} filas")

---
## 3. Decisiones de diseño y justificación

*Por qué esta arquitectura de capas, qué reglas de negocio se aplican
y qué se espera que cueste computacionalmente.*

---
## 4. Implementación
### 4.1 Herramienta de medición

*El requisito es medir tres veces y reportar la mediana. Una sola corrida es ruido:
en pruebas, un join pasó de 2,8 a 1,6 segundos, un rango en el que una medición
aislada puede dar el resultado invertido.*

In [0]:
import time, statistics

def medir(funcion, repeticiones=3, etiqueta=""):
    """Ejecuta la función varias veces y reporta la mediana."""
    tiempos = []
    for i in range(repeticiones):
        inicio = time.time()
        funcion()
        tiempos.append(time.time() - inicio)
    mediana = statistics.median(tiempos)
    print(f"{etiqueta:35s} mediana {mediana:6.2f}s   corridas {[round(t,2) for t in tiempos]}")
    return mediana

Se mide con tres corridas y se reporta la mediana en vez de una sola ejecución, porque el tiempo de una consulta en Spark varía entre corridas por factores externos al código (estado de la caché de Spark, otros procesos en el clúster compartido, JIT warm-up de la JVM). Una sola medición puede caer en cualquier punto de ese rango y llevar a una conclusión equivocada sobre si una optimización realmente ayudó; la mediana de tres corridas amortigua ese ruido sin necesitar decenas de repeticiones.

### 4.2 Capa bronce

In [0]:
CATALOGO = "bigdata_grupo63"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOGO}.tpch")

from pyspark.sql import functions as F

tablas_tpch = ["lineitem", "orders"]

for t in tablas_tpch:
    (spark.table(f"samples.tpch.{t}")
        .withColumn("_ingested_at", F.current_timestamp())
        .write.format("delta").mode("overwrite")
        .saveAsTable(f"{CATALOGO}.tpch.bronze_{t}"))
    print(f"bronze_{t} creada:", spark.table(f"{CATALOGO}.tpch.bronze_{t}").count(), "filas")

Se creó un esquema `tpch` separado de `wanderbricks` dentro del mismo catálogo (`bigdata_grupo63`), porque este es un caso de estudio distinto (benchmark TPC-H) y mezclarlo con las tablas de Wanderbricks confundiría el propósito de cada esquema. Ambas tablas (`lineitem`, `orders`) se ingieren tal como llegan, sin transformación, siguiendo el mismo principio de bronce: preservar los datos crudos para poder reprocesar desde cero si la lógica de limpieza de la capa plata cambia.

### 4.3 Capa plata — calidad de datos

*Nulos, duplicados, tipos y al menos dos reglas de negocio, con conteos antes y después.*

In [0]:
CATALOGO = "bigdata_grupo63"

bronze_li = spark.table(f"{CATALOGO}.tpch.bronze_lineitem")
bronze_or = spark.table(f"{CATALOGO}.tpch.bronze_orders")

# --- Diagnóstico de calidad (antes) ---
from pyspark.sql import functions as F

print("=== lineitem ===")
print("filas totales:", bronze_li.count())
print("duplicados exactos:", bronze_li.count() - bronze_li.dropDuplicates().count())
bronze_li.select([F.count(F.when(F.col(c).isNull(), c)).alias(c) for c in bronze_li.columns]).display()

print("=== orders ===")
print("filas totales:", bronze_or.count())
print("duplicados exactos:", bronze_or.count() - bronze_or.dropDuplicates().count())
bronze_or.select([F.count(F.when(F.col(c).isNull(), c)).alias(c) for c in bronze_or.columns]).display()

In [0]:
# --- Regla de negocio 1: descartar líneas de pedido con cantidad o precio inválidos ---
antes_li = bronze_li.count()

silver_lineitem = (bronze_li
    .filter((F.col("l_quantity") > 0) & (F.col("l_extendedprice") > 0))
    .withColumn("l_shipdate", F.col("l_shipdate").cast("date"))
    .withColumn("l_commitdate", F.col("l_commitdate").cast("date"))
    .withColumn("l_receiptdate", F.col("l_receiptdate").cast("date")))

# --- Regla de negocio 2: descartar líneas con fechas inconsistentes (recibido antes de enviado) ---
tras_regla1 = silver_lineitem.count()
silver_lineitem = silver_lineitem.filter(F.col("l_receiptdate") >= F.col("l_shipdate"))
despues_li = silver_lineitem.count()

print(f"lineitem — antes: {antes_li:,} | tras regla 1 (cantidad/precio válido): {tras_regla1:,} | tras regla 2 (fechas consistentes): {despues_li:,}")

silver_lineitem.write.format("delta").mode("overwrite").saveAsTable(f"{CATALOGO}.tpch.silver_lineitem")

In [0]:
# --- Orders: tipado y filtro de totalprice válido ---
antes_or = bronze_or.count()

silver_orders = (bronze_or
    .filter(F.col("o_totalprice") > 0)
    .withColumn("o_orderdate", F.col("o_orderdate").cast("date"))
    .withColumn("o_totalprice", F.col("o_totalprice").cast("decimal(12,2)")))

despues_or = silver_orders.count()
print(f"orders — antes: {antes_or:,} | después (totalprice válido): {despues_or:,}")

silver_orders.write.format("delta").mode("overwrite").saveAsTable(f"{CATALOGO}.tpch.silver_orders")

Se aplicaron dos reglas de negocio sobre `lineitem`, ambas con conteo antes/después documentado:

1. **Cantidad y precio válidos** (`l_quantity > 0` y `l_extendedprice > 0`): descarta líneas de pedido sin sentido económico, una línea con cantidad cero o precio cero no representa una venta real.
2. **Consistencia de fechas** (`l_receiptdate >= l_shipdate`): descarta registros donde la fecha de recepción es anterior a la de envío, una imposibilidad lógica que indica error de captura de datos.

Sobre `orders` se aplicó tipado explícito (`o_orderdate` a `date`, `o_totalprice` a `decimal(12,2)`) y un filtro de `o_totalprice > 0`, siguiendo el mismo criterio: un pedido con valor total cero o negativo no es válido para el análisis de negocio.

### 4.4 Transformaciones distribuidas

*Como mínimo un join entre tablas grandes y una agregación con función de ventana.*

In [0]:
from pyspark.sql import Window
from pyspark.sql import functions as F

CATALOGO = "bigdata_grupo63"

silver_lineitem = spark.table(f"{CATALOGO}.tpch.silver_lineitem")
silver_orders = spark.table(f"{CATALOGO}.tpch.silver_orders")

# --- Join entre las dos tablas grandes ---
lineitem_con_orden = (silver_lineitem
    .join(silver_orders, silver_lineitem.l_orderkey == silver_orders.o_orderkey)
    .withColumn("revenue_linea", F.col("l_extendedprice") * (1 - F.col("l_discount"))))

print("filas tras el join:", lineitem_con_orden.count())

# --- Agregación con función de ventana ---
# Ranking de pedidos por ingreso total, dentro de cada nivel de prioridad de pedido
ingresos_por_orden = (lineitem_con_orden
    .groupBy("o_orderkey", "o_orderpriority")
    .agg(F.sum("revenue_linea").alias("ingreso_total_orden")))

ventana = Window.partitionBy("o_orderpriority").orderBy(F.desc("ingreso_total_orden"))

ranking_por_prioridad = (ingresos_por_orden
    .withColumn("ranking_dentro_prioridad", F.rank().over(ventana)))

ranking_por_prioridad.filter(F.col("ranking_dentro_prioridad") <= 5).display()

El join cruza `silver_lineitem` (~30M filas) con `silver_orders` (~7.5M filas) por `orderkey`, calculando el ingreso de cada línea de pedido (`revenue_linea = extendedprice × (1 - discount)`), la fórmula estándar de ingreso neto de TPC-H, que descuenta el porcentaje de descuento aplicado a cada línea.

Sobre el resultado agregado por pedido se aplicó una función de ventana (`RANK() OVER (PARTITION BY o_orderpriority ORDER BY ingreso_total_orden DESC)`), que rankea los pedidos por ingreso total **dentro de cada nivel de prioridad** (`o_orderpriority`), sin colapsar los datos en una sola fila por grupo como haría un `GROUP BY`

### 4.5 Medición base

*Antes de optimizar nada.*

In [0]:
from pyspark.sql import Window
from pyspark.sql import functions as F

CATALOGO = "bigdata_grupo63"

def consulta_base():
    silver_lineitem = spark.table(f"{CATALOGO}.tpch.silver_lineitem")
    silver_orders = spark.table(f"{CATALOGO}.tpch.silver_orders")

    resultado = (silver_lineitem
        .join(silver_orders, silver_lineitem.l_orderkey == silver_orders.o_orderkey)
        .withColumn("revenue_linea", F.col("l_extendedprice") * (1 - F.col("l_discount")))
        .groupBy("o_orderkey", "o_orderpriority")
        .agg(F.sum("revenue_linea").alias("ingreso_total_orden")))

    resultado.count()  # acción que fuerza la ejecución completa

t_base = medir(consulta_base, etiqueta="Base (join + agregación)")

In [0]:
# Plan de ejecución ANTES de optimizar
silver_lineitem = spark.table(f"{CATALOGO}.tpch.silver_lineitem")
silver_orders = spark.table(f"{CATALOGO}.tpch.silver_orders")

df_base = (silver_lineitem
    .join(silver_orders, silver_lineitem.l_orderkey == silver_orders.o_orderkey)
    .groupBy("o_orderkey")
    .agg(F.sum(F.col("l_extendedprice") * (1 - F.col("l_discount"))).alias("ingreso_total_orden")))

df_base.explain(mode="formatted")

La medición sobre tres corridas dio una mediana de **2.00s** (corridas: 2.48s, 1.97s, 2.00s) para el join entre `silver_lineitem` ( ~30M filas) y `silver_orders` ( ~7.5M filas) seguido de la agregación por `orderkey`.

El plan de ejecución **usa `PhotonShuffledHashJoin`**, Dos detalles del plan que son relevantes:

1. **Poda de columnas automática**: el escaneo de `silver_lineitem` solo lee `l_orderkey`, `l_extendedprice` y `l_discount`, y el de `silver_orders` solo lee `o_orderkey`, el optimizador Catalyst detectó que el resto de columnas no se usan en esta consulta y las descarta antes de mover un solo byte por la red.
2. **Agregación parcial antes del shuffle**: Spark ejecuta un `PhotonGroupingAgg` (suma parcial por `orderkey`) sobre `lineitem` **antes** de barajar los datos, y solo después hace el shuffle y el join. Esto reduce drásticamente el volumen de datos que viaja por la red, en vez de mover las ~30M filas crudas de `lineitem`, mueve una fila por cada `orderkey` distinto con su suma parcial ya calculada.

### 4.6 Optimización

*Al menos dos técnicas: particionado o liquid clustering, broadcast join, caché,
reescritura de la consulta, filtros anticipados.*

In [0]:
spark.sql(f"OPTIMIZE {CATALOGO}.tpch.silver_lineitem ZORDER BY (l_orderkey)")

from pyspark.sql import functions as F

CATALOGO = "bigdata_grupo63"

# Técnica 1: Broadcast join explícito
# Técnica 2: Poda de columnas explícita antes del join (en vez de dejarlo al optimizador)

def consulta_optimizada():
    li = spark.table(f"{CATALOGO}.tpch.silver_lineitem").select("l_orderkey", "l_extendedprice", "l_discount")
    orders = spark.table(f"{CATALOGO}.tpch.silver_orders").select("o_orderkey")

    resultado = (li
        .join(F.broadcast(orders), li.l_orderkey == orders.o_orderkey)
        .withColumn("revenue_linea", F.col("l_extendedprice") * (1 - F.col("l_discount")))
        .groupBy("o_orderkey")
        .agg(F.sum("revenue_linea").alias("ingreso_total_orden")))
    resultado.count()

t_opt = medir(consulta_optimizada, etiqueta="Optimizada (broadcast + Z-order)")
print(f"\nMejora: {(1 - t_opt/t_base)*100:.1f}%")

In [0]:
# Plan de ejecución DESPUÉS de optimizar
li = spark.table(f"{CATALOGO}.tpch.silver_lineitem").select("l_orderkey", "l_extendedprice", "l_discount")
orders = spark.table(f"{CATALOGO}.tpch.silver_orders").select("o_orderkey")

df_opt = (li
    .join(F.broadcast(orders), li.l_orderkey == orders.o_orderkey)
    .groupBy("o_orderkey")
    .agg(F.sum(F.col("l_extendedprice") * (1 - F.col("l_discount"))).alias("ingreso_total_orden")))

df_opt.explain(mode="formatted")

**Qué cambió en el plan físico:**

*Identificar el cambio concreto — por ejemplo, de `SortMergeJoin` a `BroadcastHashJoin` —
y explicar por qué eso mejora el tiempo **en este caso**. No basta con decir que quedó
más rápido: hay que decir qué hizo el motor distinto.*

Se aplicaron dos técnicas — *broadcast join* explícito y `OPTIMIZE ZORDER BY (l_orderkey)` — ajustadas a las restricciones del compute serverless (sin `cache()`/`persist()` ni `spark.conf.set`).

**Resultado medido:** la versión "optimizada" dio una mediana de 2.28s frente a 2.00s de la base — una **mejora negativa de -13.7%**. El plan de ejecución confirma que el broadcast sí ocurrió (`PhotonShuffledHashJoin` → `PhotonBroadcastHashJoin`, con `orders` distribuida vía `EXECUTOR_BROADCAST` en vez de shuffle), pero el resultado real fue más lento, por tres razones: (1) sin caché, el costo de recolectar y distribuir el broadcast se paga en cada una de las tres corridas, sin poder amortizarse; (2) el plan optimizado conserva un shuffle en la agregación final por `o_orderkey`, así que solo se eliminó uno de los dos cuellos de botella; (3) el `ZORDER` no aporta nada porque la consulta no tiene ningún filtro selectivo sobre `l_orderkey`, lee la tabla completa de igual forma con o sin reorganización física.

Este resultado es evidencia directa de que una técnica de optimización no es universalmente buena: el broadcast join rinde cuando la tabla difundida se reutiliza entre múltiples consultas,algo que no pud0 aprovechar por la restricción de serverless y cuando reemplaza un shuffle costoso frente a una consulta lo bastante pesada como para que ese ahorro compense el overhead de distribución, ninguna de las dos condiciones se cumplió en este caso concreto.

### 4.7 Capa oro y consumo

*Tabla oro o dashboard que responda tres preguntas concretas del negocio.*

In [0]:
from pyspark.sql import functions as F

CATALOGO = "bigdata_grupo63"

silver_lineitem = spark.table(f"{CATALOGO}.tpch.silver_lineitem")
silver_orders = spark.table(f"{CATALOGO}.tpch.silver_orders")

# Tabla oro: resumen por pedido, con métricas listas para consumo directo
gold_ordenes = (silver_lineitem
    .withColumn("revenue_linea", F.col("l_extendedprice") * (1 - F.col("l_discount")))
    .withColumn("entrega_tardia", (F.col("l_receiptdate") > F.col("l_commitdate")).cast("int"))
    .groupBy("l_orderkey")
    .agg(
        F.sum("revenue_linea").alias("ingreso_total"),
        F.count("*").alias("num_lineas"),
        F.sum("entrega_tardia").alias("lineas_tardias")
    )
    .join(silver_orders, silver_lineitem.l_orderkey == silver_orders.o_orderkey, "inner")
    .select(
        "o_orderkey", "o_orderdate", "o_orderpriority",
        "ingreso_total", "num_lineas", "lineas_tardias"
    )
    .withColumn("anio_pedido", F.year("o_orderdate"))
    .withColumn("pct_lineas_tardias", F.round(F.col("lineas_tardias") / F.col("num_lineas") * 100, 1))
)

gold_ordenes.write.format("delta").mode("overwrite").saveAsTable(f"{CATALOGO}.gold.resumen_ordenes")
print("gold.resumen_ordenes creada:", spark.table(f"{CATALOGO}.gold.resumen_ordenes").count(), "filas")

---
## 5. Resultados

| Pregunta del negocio | Respuesta obtenida |
|---|---|
| ¿Cómo evoluciona el ingreso total por año de pedido? | El ingreso anual se mantuvo estable entre 1992 y 1997 (~165 mil millones cada año), sin tendencia de crecimiento ni decrecimiento. La caída a ~97 mil millones en 1998 no es una señal real de negocio: corresponde a que el dataset de TPC-H solo cubre pedidos hasta mediados de 1998, es decir, es un año parcial, no una contracción del negocio. |
| ¿Qué nivel de prioridad concentra más ingresos? | Ninguno de forma significativa: las cinco categorías de prioridad (`1-URGENT` a `5-LOW`, incluyendo `4-NOT SPECIFIED`) tienen ingresos casi idénticos ( ~218 mil millones cada una) y un número de pedidos prácticamente igual ( ~1.5M cada una). La prioridad declarada del pedido no está correlacionada con su valor económico. |
| ¿Varía la tasa de entregas tardías según la prioridad del pedido? | No, la tasa de líneas con entrega tardía es exactamente **63.2% en las cinco categorías de prioridad**, incluida `1-URGENT`. Los pedidos marcados como urgentes no se entregan mejor ni peor que el resto. |


In [0]:
# ¿Cómo evoluciona el ingreso total por año de pedido?

(spark.table(f"{CATALOGO}.gold.resumen_ordenes")
    .groupBy("anio_pedido")
    .agg(F.sum("ingreso_total").alias("ingreso_anual"))
    .orderBy("anio_pedido")
    .display())

In [0]:
# ¿Qué nivel de prioridad concentra más ingresos?
(spark.table(f"{CATALOGO}.gold.resumen_ordenes")
    .groupBy("o_orderpriority")
    .agg(
        F.sum("ingreso_total").alias("ingreso_total_prioridad"),
        F.count("*").alias("num_pedidos")
    )
    .orderBy(F.desc("ingreso_total_prioridad"))
    .display())

In [0]:
# ¿Qué porcentaje de líneas de pedido llegan tarde, y varía según la prioridad del pedido?
(spark.table(f"{CATALOGO}.gold.resumen_ordenes")
    .groupBy("o_orderpriority")
    .agg(
        F.sum("lineas_tardias").alias("total_lineas_tardias"),
        F.sum("num_lineas").alias("total_lineas"),
    )
    .withColumn("pct_tardias", F.round(F.col("total_lineas_tardias") / F.col("total_lineas") * 100, 1))
    .orderBy(F.desc("pct_tardias"))
    .display())

---
## 6. Conclusiones

*Qué funcionó, qué no funcionó y qué harían distinto si empezaran de nuevo.
Las conclusiones deben derivarse de los resultados mostrados arriba, no de expectativas generales.*

Este ejercicio me dejó dos aprendizajes que van más allá de la sintaxis de Spark. El primero es metodológico: la optimización salió **contraproducente** (2.28s vs. 2.00s base, -13.7%), y ese resultado solo se pudo detectar y explicar porque se midió con tres corridas y se comparó el plan de ejecución antes/después en vez de asumir que "broadcast join siempre mejora". El plan confirmó que la estrategia de join cambió como se esperaba (`PhotonShuffledHashJoin` → `PhotonBroadcastHashJoin`), pero el resultado real fue peor porque el compute serverless no permite cachear la tabla difundida, así que su costo de distribución se pagó en cada corrida sin poder amortizarse y porque la consulta ya era tan rápida (2s) que el overhead de preparar el broadcast pesó más que el shuffle que reemplazaba. La lección no es "el broadcast join es malo", sino que ninguna técnica de optimización es universal: hay que medir, no asumir.

El segundo aprendizaje es de calidad de datos, y salió de la capa oro: la tasa de entregas tardías fue idéntica (63.2%) en las cinco categorías de `o_orderpriority`, incluyendo los pedidos marcados como urgentes. Si `lineitem` fuera un dataset de un negocio real, esto sería una alerta operativa seria — significaría que la prioridad del pedido no influye en absoluto en qué tan bien se cumple su entrega. Aquí se explica por la naturaleza sintética de TPC-H, pero el ejercicio de detectarlo y saber diferenciarlo de una señal real es el mismo que se aplicaría en un caso de negocio genuino.

**Qué se haría distinto:** si el compute lo permitiera, se repetiría la medición con caché habilitada para aislar cuánto del costo del broadcast corresponde a la distribución en sí y cuánto a la falta de reutilización  sin eso, no se puede afirmar con certeza si el broadcast join sería mejor en un clúster clásico con las mismas tablas. También se probaría una tercera técnica (filtros anticipados o reescritura de la consulta) para tener una comparación más completa antes de concluir que ninguna optimización ayuda en este caso específico.


---
## 7. Reparto del trabajo y uso de IA

| Integrante | De qué se encargó | Qué sustenta en el video |
|---|---|---|
| Jorge Andrés Ocampo Suárez | Todas las sesiones | Todas las sesiones |
| | | |
| | | |

**Uso de asistentes de IA:** *(indicar en qué partes se usó el Databricks Assistant u otra
herramienta. Está permitido; lo que se evalúa es que cada integrante pueda explicar
cualquier línea del código en el video.)*

> Este reparto debe coincidir con lo que cada persona demuestra en el video y con el
> historial de commits del repositorio. Las tres fuentes se contrastan al calificar.

---
## 📹 Preguntas obligatorias de sustentación

Cada integrante responde estas tres preguntas en su intervención del video:

1. Muestre el plan de ejecución antes y después de optimizar e indique qué cambió.
2. ¿Por qué esa técnica mejoró el tiempo en este caso concreto?
3. ¿En qué situación su optimización dejaría de servir o sería contraproducente?

---
## ✅ Antes de entregar

- [ ] El volumen mínimo está verificado y visible en la salida
- [ ] Las mediciones tienen tres corridas y se reporta la mediana
- [ ] Los planes de ejecución antes y después están en el notebook
- [ ] El video muestra ambos planes — es obligatorio en esta evidencia
- [ ] La capa oro responde las tres preguntas del negocio
- [ ] Todo confirmado en /ea3 y el HTML subido a Canvas